# 3. Эксперименты с ML

Задача: классифицировать пару как совпадение или несовпадение по семи рассчитанным признакам.
Используются 882 размеченные пары: 436 совпадений и 446 несовпадений.

Сравниваются Logistic Regression, Random Forest и HistGradientBoosting. Порог выбирается
по правилу precision ≥ 95%, затем максимальный recall. Это исследовательский эксперимент,
а не подтверждение 95% точности на всём неразмеченном каталоге.

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Работает при запуске Jupyter из корня проекта и из папки notebooks.
ROOT = Path.cwd()
if not (ROOT / 'data' / 'processed').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'data' / 'processed').is_dir():
    raise FileNotFoundError('Открой Jupyter из корня проекта.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


## Проверка качества

5-fold `StratifiedGroupKFold`, группы — товары Sklavenitis, seed = 42.
Пары одного товара Sklavenitis не попадают одновременно в обучение и проверку.
Товары AlphaMega могут повторяться между folds; независимой финальной тестовой выборки нет.
Готовый суммарный скоринг в признаки не включён.

In [2]:
from scripts.ml_experiments import evaluate_models
cv_results, oof_predictions, selected_policy = evaluate_models()
best = (cv_results.loc[cv_results['precision'] >= .95]
        .sort_values(['recall', 'precision'], ascending=False)
        .groupby('model_name', sort=False).head(1))
best[['model_name', 'threshold', 'precision', 'recall', 'f1', 'roc_auc']]

Training rows: 882 groups: 591
{
  "model_name": "logistic_regression",
  "threshold": 0.7,
  "roc_auc": 0.9475948286501831,
  "accuracy": 0.8707482993197279,
  "precision": 0.952247191011236,
  "recall": 0.7775229357798165,
  "f1": 0.8560606060606061,
  "tp": 339,
  "fp": 17,
  "tn": 429,
  "fn": 97,
  "predicted_match_count": 356
}


,model_name,threshold,precision,recall,f1,roc_auc
12,logistic_regression,0.7,0.952247,0.777523,0.856061,0.947595
50,hist_gradient_boosting,0.8,0.957020,0.766055,0.850955,0.952614
30,random_forest,0.7,0.954286,0.766055,0.849873,0.952089


## Исторический результат и повторный запуск

Историческая таблица сохранена отдельно: `data/reference/historical_ml_cv.csv`.
В исходном окружении при пороге 0,75 выбирался Random Forest: precision 96,22%, recall 75,92%.
Первоначальные версии всех библиотек не были зафиксированы. Сопоставляем повторный запуск
с сохранёнными метриками; расхождения не перезаписывают исторический результат.

In [3]:
comparison = pd.read_csv(ROOT / 'reports/tables/ml_reproduction_comparison.csv')
comparison.loc[(comparison['model_name'] == 'random_forest') & (comparison['threshold'] == .75),
    ['model_name', 'threshold', 'precision_historical', 'precision_rerun',
     'recall_historical', 'recall_rerun']]

,model_name,threshold,precision_historical,precision_rerun,recall_historical,recall_rerun
31,random_forest,0.75,0.962209,0.958824,0.759174,0.747706


## Что показала ручная проверка

По мере снижения ML-вероятности автор не наблюдала столь же понятного ухудшения совпадений,
как при снижении взвешенного скоринга. Для последующей проверки и доразметки скоринг оказался
полезнее. Итоговая выборка объединяет ручную разметку, принятые ML-совпадения и ручное разрешение дубликатов.

Это качественное наблюдение. Отдельного эксперимента, доказывающего количественное превосходство
скоринга над ML, не проводилось. Размеченные пары выбирались из диапазонов скоринга и имеют
почти равный баланс классов; на всём пуле кандидатов доля совпадений другая. Поэтому CV precision
не гарантирует такое же качество при применении к неразмеченным парам.

## Необязательное предсказание для всего пула

Требует запуска ноутбука 02. Результат сохраняется отдельно и не заменяет финальные проверенные пары.

In [4]:
RUN_FULL_PREDICTION = False
if RUN_FULL_PREDICTION:
    from scripts.ml_experiments import predict_candidates
    new_predictions = predict_candidates()
    display(new_predictions[['item_id_sk', 'item_id_am', 'ml_match_probability']].head())